# Data Validation

This notebook validates the raw Vietnam history CSV files before dashboard use. It checks schema, primary keys, foreign-key style references, year ranges, score ranges, and coordinate ranges. Validation outputs are written to `data/processed/`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'raw' / 'events.csv').exists():
            return candidate
    raise FileNotFoundError('Could not find project root containing data/raw/events.csv')

ROOT = find_project_root()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    'events': RAW_DIR / 'events.csv',
    'people': RAW_DIR / 'people.csv',
    'periods': RAW_DIR / 'periods.csv',
    'locations': RAW_DIR / 'locations.csv',
}

EXPECTED_COLUMNS = {
    'events': ['event_id', 'event_name', 'year_start', 'year_end', 'year_label', 'date_label', 'date_detail', 'period_id', 'period_name', 'broad_era', 'category', 'location_ids', 'location_names', 'person_ids', 'person_names', 'description_short', 'description_long', 'importance_score'],
    'people': ['person_id', 'person_name', 'other_names', 'role', 'person_category', 'related_period_id', 'related_period_name', 'first_mentioned_year', 'last_mentioned_year', 'birth_year', 'death_year', 'age_at_death_estimate', 'importance_score'],
    'periods': ['period_id', 'period_name', 'start_year', 'end_year', 'start_label', 'end_label', 'broad_era', 'main_capital', 'period_type', 'description', 'date_status'],
    'locations': ['location_id', 'location_name', 'modern_name', 'location_type', 'modern_country', 'modern_region', 'latitude', 'longitude', 'first_mentioned_year', 'related_period_ids'],
}

ID_COLUMNS = {
    'events': 'event_id',
    'people': 'person_id',
    'periods': 'period_id',
    'locations': 'location_id',
}

In [ ]:
data = {}
for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(path)
    data[name] = pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')

{name: df.shape for name, df in data.items()}

In [ ]:
def split_ids(value):
    value = str(value).strip()
    if not value:
        return []
    return [part.strip() for part in value.replace('|', ';').split(';') if part.strip()]

def to_number(series):
    return pd.to_numeric(series.replace('', pd.NA), errors='coerce')

issues = []

def add_issue(severity, dataset, check, row_id='', column='', value='', message=''):
    issues.append({
        'severity': severity,
        'dataset': dataset,
        'check': check,
        'row_id': row_id,
        'column': column,
        'value': value,
        'message': message,
    })

In [ ]:
# Schema and primary-key checks
for name, expected in EXPECTED_COLUMNS.items():
    df = data[name]
    missing_columns = [col for col in expected if col not in df.columns]
    extra_columns = [col for col in df.columns if col not in expected]
    for col in missing_columns:
        add_issue('error', name, 'schema', column=col, message='Required column is missing.')
    for col in extra_columns:
        add_issue('warning', name, 'schema', column=col, message='Unexpected column is present.')

    id_col = ID_COLUMNS[name]
    blank_ids = df[df[id_col].str.strip().eq('')]
    for _, row in blank_ids.iterrows():
        add_issue('error', name, 'primary_key', column=id_col, message='Blank primary key.')

    duplicate_ids = df[df[id_col].duplicated(keep=False)]
    for _, row in duplicate_ids.iterrows():
        add_issue('error', name, 'primary_key', row_id=row[id_col], column=id_col, value=row[id_col], message='Duplicate primary key.')

In [ ]:
# Reference checks
period_ids = set(data['periods']['period_id'])
person_ids = set(data['people']['person_id'])
location_ids = set(data['locations']['location_id'])
period_names = dict(zip(data['periods']['period_id'], data['periods']['period_name']))

for _, row in data['events'].iterrows():
    if row['period_id'] not in period_ids:
        add_issue('error', 'events', 'period_reference', row['event_id'], 'period_id', row['period_id'], 'Event period_id does not exist in periods.csv.')
    elif row['period_name'] and row['period_name'] != period_names.get(row['period_id']):
        add_issue('warning', 'events', 'period_name_match', row['event_id'], 'period_name', row['period_name'], 'Event period_name does not match periods.csv for the same period_id.')

    for person_id in split_ids(row['person_ids']):
        if person_id not in person_ids:
            add_issue('error', 'events', 'person_reference', row['event_id'], 'person_ids', person_id, 'Referenced person_id does not exist in people.csv.')

    for location_id in split_ids(row['location_ids']):
        if location_id not in location_ids:
            add_issue('error', 'events', 'location_reference', row['event_id'], 'location_ids', location_id, 'Referenced location_id does not exist in locations.csv.')

for _, row in data['people'].iterrows():
    if row['related_period_id'] and row['related_period_id'] not in period_ids:
        add_issue('error', 'people', 'period_reference', row['person_id'], 'related_period_id', row['related_period_id'], 'related_period_id does not exist in periods.csv.')

for _, row in data['locations'].iterrows():
    for period_id in split_ids(row['related_period_ids']):
        if period_id not in period_ids:
            add_issue('error', 'locations', 'period_reference', row['location_id'], 'related_period_ids', period_id, 'related_period_ids contains an ID that does not exist in periods.csv.')

In [ ]:
# Numeric, date, score, and coordinate checks
for dataset, cols in {
    'events': ['year_start', 'year_end', 'importance_score'],
    'people': ['first_mentioned_year', 'last_mentioned_year', 'birth_year', 'death_year', 'age_at_death_estimate', 'importance_score'],
    'periods': ['start_year', 'end_year'],
    'locations': ['latitude', 'longitude', 'first_mentioned_year'],
}.items():
    df = data[dataset]
    id_col = ID_COLUMNS[dataset]
    for col in cols:
        numeric = to_number(df[col])
        bad = df[df[col].str.strip().ne('') & numeric.isna()]
        for _, row in bad.iterrows():
            add_issue('error', dataset, 'numeric_parse', row[id_col], col, row[col], 'Value should be numeric or blank.')

events = data['events'].copy()
event_start = to_number(events['year_start'])
event_end = to_number(events['year_end'])
bad_ranges = events[event_start.notna() & event_end.notna() & (event_end < event_start)]
for _, row in bad_ranges.iterrows():
    add_issue('error', 'events', 'date_range', row['event_id'], 'year_end', row['year_end'], 'year_end is earlier than year_start.')

periods = data['periods'].copy()
p_start = to_number(periods['start_year'])
p_end = to_number(periods['end_year'])
for _, row in periods[p_start.notna() & p_end.notna() & (p_end < p_start)].iterrows():
    add_issue('error', 'periods', 'date_range', row['period_id'], 'end_year', row['end_year'], 'end_year is earlier than start_year.')

for dataset in ['events', 'people']:
    df = data[dataset]
    id_col = ID_COLUMNS[dataset]
    scores = to_number(df['importance_score'])
    invalid_scores = df[df['importance_score'].str.strip().ne('') & ~scores.between(1, 5)]
    for _, row in invalid_scores.iterrows():
        add_issue('error', dataset, 'importance_score', row[id_col], 'importance_score', row['importance_score'], 'importance_score should be between 1 and 5.')

locations = data['locations']
lat = to_number(locations['latitude'])
lon = to_number(locations['longitude'])
bad_coords = locations[(locations['latitude'].str.strip().ne('') & ~lat.between(-90, 90)) | (locations['longitude'].str.strip().ne('') & ~lon.between(-180, 180))]
for _, row in bad_coords.iterrows():
    add_issue('error', 'locations', 'coordinate_range', row['location_id'], 'latitude/longitude', f"{row['latitude']}, {row['longitude']}", 'Coordinates are outside valid latitude/longitude ranges.')

In [ ]:
# Missingness summary
summary_rows = []
for name, df in data.items():
    id_col = ID_COLUMNS[name]
    summary_rows.append({
        'dataset': name,
        'rows': len(df),
        'columns': len(df.columns),
        'id_column': id_col,
        'unique_ids': df[id_col].nunique(),
        'blank_ids': int(df[id_col].str.strip().eq('').sum()),
        'duplicate_id_rows': int(df[id_col].duplicated(keep=False).sum()),
    })

validation_summary = pd.DataFrame(summary_rows)
validation_issues = pd.DataFrame(issues, columns=['severity', 'dataset', 'check', 'row_id', 'column', 'value', 'message'])

validation_summary

In [ ]:
missingness = []
for name, df in data.items():
    for col in df.columns:
        missingness.append({
            'dataset': name,
            'column': col,
            'missing_count': int(df[col].astype(str).str.strip().eq('').sum()),
            'missing_percent': round(float(df[col].astype(str).str.strip().eq('').mean() * 100), 2),
        })
missingness = pd.DataFrame(missingness).sort_values(['dataset', 'missing_count'], ascending=[True, False])
missingness.head(20)

In [ ]:
validation_summary.to_csv(PROCESSED_DIR / 'validation_summary.csv', index=False, encoding='utf-8-sig')
validation_issues.to_csv(PROCESSED_DIR / 'validation_issues.csv', index=False, encoding='utf-8-sig')
missingness.to_csv(PROCESSED_DIR / 'validation_missingness.csv', index=False, encoding='utf-8-sig')

report = {
    'datasets': validation_summary.to_dict(orient='records'),
    'issue_counts': validation_issues['severity'].value_counts().to_dict() if not validation_issues.empty else {},
    'output_files': [
        'validation_summary.csv',
        'validation_issues.csv',
        'validation_missingness.csv',
    ],
}
(PROCESSED_DIR / 'validation_report.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

print(f"Validation issues found: {len(validation_issues)}")
validation_issues.head(20)